# P3 - ResNet-18 Fine-Tuning 

Deep Learning Transfer Learning pour détection de métastases CAMELYON17

**Auteur** : Rosane
**Branche** : P3-Resnet

#  Imports & Configuration

Setup device (GPU/CPU), seed, et hyperparamètres

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import models
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 10
SEED = 42

# Fixer seed pour reproductibilité
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

print(f"Device: {DEVICE}")
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"\n OK - Configuration ready")

# les Données

In [ ]:
import os, sys
from google.colab import drive
drive.mount("/content/drive")

if not os.path.exists("depot"):
    !git clone --depth 1 https://github.com/kenzo-kouokam/detection-metastases-camelyon17.git depot
sys.path.insert(0, "depot/p1_data")
from charger_donnees import charger_donnees

df = charger_donnees()
print(f"Total patches: {len(df)}")
print(f"\nDistribution par split:")
print(df['split_equipe'].value_counts())
print(f"\nDistribution labels par split:")
print(df.groupby(['split_equipe', 'label']).size().unstack())
print(f"\n OK - Données chargées")

# Architecture ResNet-18

- Backbone gelé (poids ImageNet)
- Tête remplacée : 512 → 128 → 2 classes
- Fine-tune tête seulement

In [ ]:
# Charger ResNet-18 pré-entraîné
print("Chargement ResNet-18...")
model = models.resnet18(pretrained=True)

# Geler le backbone (features layers)
for param in model.parameters():
    param.requires_grad = False

# Remplacer la tête (classification layers)
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_features, 128),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 2)  # 2 classes: normal (0) vs tumoral (1)
)

# Dégeler la tête (fine-tuning)
for param in model.fc.parameters():
    param.requires_grad = True

# Placer modèle sur device
model = model.to(DEVICE)

# Vérifier paramètres
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Model Architecture:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Backbone frozen: {total_params - trainable_params:,}")
print(f"\n OK - ResNet-18 ready")

# Vérification 

In [ ]:
# Créer batch test
test_batch_size = 4
test_images = torch.randn(test_batch_size, 3, 224, 224).to(DEVICE)
test_labels = torch.tensor([0, 1, 0, 1]).to(DEVICE)

# Forward pass
with torch.no_grad():
    outputs = model(test_images)
    probas = torch.softmax(outputs, dim=1)

# Calculer loss test
criterion = nn.CrossEntropyLoss()
test_loss = criterion(outputs, test_labels)

print(f"Test batch shape: {test_images.shape}")
print(f"Model outputs shape: {outputs.shape}")
print(f"Test loss: {test_loss.item():.4f}")
print(f"\n OK - Forward pass verified")

# Tensors PyTorch

Convertir 39k images du parquet en tensors PyTorch normalisés (3, 224, 224)

In [ ]:
import cv2
from io import BytesIO
import time

print("="*60)
print(" Préparer Tensors PyTorch")
print("="*60)

# Récharger les données avec images
print("\nChargement des 39k images du parquet...")
df_images = charger_donnees()
images_dict = df_images.set_index('id_patch')['image'].to_dict()
print(f" {len(images_dict)} images chargées")

# Préparer les splits
splits = ['train_interne', 'val_interne', 'val_ood', 'test']
data_splits = {}

print(f"\nPréparation des splits...")
for split in splits:
    split_df = df_images[df_images['split_equipe'] == split].copy()
    data_splits[split] = split_df
    print(f"  {split}: {len(split_df)} patchs ({split_df['label'].sum()} tumoraux)")

# Fonction pour charger et normaliser image
def load_and_process_image(image_data, size=224):
    """Charger image bytes, redimensionner et normaliser"""
    if isinstance(image_data, bytes):
        image_array = np.frombuffer(image_data, dtype=np.uint8)
        image = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
    else:
        image = image_data
    
    if image is None:
        return None
    
    # Redimensionner 224×224
    image = cv2.resize(image, (size, size))
    
    # RGB (OpenCV charge BGR)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Normaliser [0, 1]
    image = image.astype(np.float32) / 255.0
    
    # (H, W, C) → (C, H, W) PyTorch
    image = np.transpose(image, (2, 0, 1))
    
    return image

# Créer tensors par split
print(f"\nConversion en tensors PyTorch...")
tensors_dict = {}
labels_dict = {}

start_time = time.time()
for split in splits:
    print(f"  {split}...", end="", flush=True)
    split_df = data_splits[split]
    
    images_list = []
    labels_list = []
    
    for idx, row in split_df.iterrows():
        try:
            image = load_and_process_image(images_dict[row['id_patch']])
            if image is not None:
                images_list.append(image)
                labels_list.append(row['label'])
        except:
            continue
    
    # Créer tensors
    images_tensor = torch.from_numpy(np.array(images_list)).float()
    labels_tensor = torch.from_numpy(np.array(labels_list)).long()
    
    tensors_dict[split] = images_tensor
    labels_dict[split] = labels_tensor
    
    print(f" ({images_tensor.shape})")

elapsed = time.time() - start_time
print(f"\nTemps total: {elapsed:.1f}s")

# Créer DataLoaders
print(f"\nCréation des DataLoaders...")
train_dataset = TensorDataset(tensors_dict['train_interne'], labels_dict['train_interne'])
val_dataset = TensorDataset(tensors_dict['val_interne'], labels_dict['val_interne'])
val_ood_dataset = TensorDataset(tensors_dict['val_ood'], labels_dict['val_ood'])
test_dataset = TensorDataset(tensors_dict['test'], labels_dict['test'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
val_ood_loader = DataLoader(val_ood_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"  Train loader: {len(train_loader)} batches")
print(f"  Val loader: {len(val_loader)} batches")
print(f"  Val OOD loader: {len(val_ood_loader)} batches")
print(f"  Test loader: {len(test_loader)} batches")

print(f"\n OK - Tensors prêts")

# Entraînement ResNet-18 (2-3h GPU)

- Training loop 10 epochs
- Validation après chaque epoch
- Sauvegarde best_model.pth
- Plots loss/AUC

In [ ]:
print("="*60)
print("Entraîner ResNet-18")
print("="*60)

# Setup optimizer et criterion
optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

# Suivi des métriques
history = {
    'train_loss': [],
    'val_auc': [],
    'val_recall': [],
    'val_ood_auc': []
}

best_val_auc = 0.0
best_model_state = None

# Fonction d'évaluation
def evaluate(model, loader, device):
    model.eval()
    y_true = []
    y_pred_proba = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probas = torch.softmax(outputs, dim=1)[:, 1]
            
            y_true.extend(labels.cpu().numpy())
            y_pred_proba.extend(probas.cpu().numpy())
    
    y_true = np.array(y_true)
    y_pred_proba = np.array(y_pred_proba)
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    auc = roc_auc_score(y_true, y_pred_proba)
    recall = recall_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    return auc, recall, precision, f1

# Training loop
print(f"\nEntraînement sur {NUM_EPOCHS} epochs...\n")
start_training = time.time()

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    train_loss = 0.0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    history['train_loss'].append(train_loss)
    
    # Validation
    val_auc, val_recall, val_precision, val_f1 = evaluate(model, val_loader, DEVICE)
    val_ood_auc, _, _, _ = evaluate(model, val_ood_loader, DEVICE)
    
    history['val_auc'].append(val_auc)
    history['val_recall'].append(val_recall)
    history['val_ood_auc'].append(val_ood_auc)
    
    # Scheduler
    scheduler.step(val_auc)
    
    # Sauvegarder best model
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Loss: {train_loss:.4f}, Val AUC: {val_auc:.4f}* (NEW BEST)")
    else:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Loss: {train_loss:.4f}, Val AUC: {val_auc:.4f}")
    
    print(f"  Val Recall: {val_recall:.4f}, Val OOD AUC: {val_ood_auc:.4f}")

elapsed_training = time.time() - start_training
print(f"\nEntraînement terminé en {elapsed_training:.1f}s ({elapsed_training/60:.1f}min)")

# Charger best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"Chargement best_model (Val AUC: {best_val_auc:.4f})")

# Sauvegarder
torch.save(best_model_state, 'best_model.pth')
print(f" Modèle sauvegardé: best_model.pth")

# Courbes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], 'o-', label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['val_auc'], 'o-', label='Val AUC')
axes[1].plot(history['val_ood_auc'], 's-', label='Val OOD AUC')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].set_title('Validation AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=100, bbox_inches='tight')
print(f" Courbes sauvegardées: training_curves.png")
plt.show()

# Résultats finaux
test_auc, test_recall, test_precision, test_f1 = evaluate(model, test_loader, DEVICE)

print(f"\n" + "="*60)
print("OK - Entraînement complété")
print("="*60)
print(f"\nRésultats finaux:")
print(f"  Val AUC: {best_val_auc:.4f}")
print(f"  Test AUC: {test_auc:.4f}")
print(f"  Test Recall: {test_recall:.4f}")
print(f"  Test Precision: {test_precision:.4f}")
print(f"  Test F1: {test_f1:.4f}")